# 🔍 Phase 1: Dropped Row Isolation – Recovery Pipeline Setup

## 🎯 Objective

The goal of Phase 1 is to **identify and recover all rows** that were dropped during the structured parsing of the original HEARDS dataset. These dropped rows contain potentially valuable information that was not extracted using the initial regex/ML pipeline. We aim to recover and reintegrate them using a LLM-based approach (DeepSeek API).

---

## 📁 Files Used

- `heards_full_dataset.csv`: The raw, unstructured dataset.
- `heards_structured_cleaned_03.csv`: The cleaned dataset after regex/ML parsing.

---

## 🛠️ Steps Performed

1. **Uploaded both CSV files** to Colab.
2. **Loaded datasets** into pandas DataFrames (`original_df`, `cleaned_df`).
3. **Identified dropped rows** using a comparison on the `headline` column.
4. **Calculated drop statistics**:
   - Total rows dropped: `37,043`
   - Drop rate: `17.2%`
5. **Exported the dropped rows** to a new CSV: `heards_dropped_rows.csv`.

---

## 🧠 Next Step (Phase 2 Preview)

We will now move to **Phase 2: LLM-based Recovery**, where we:
- Use the DeepSeek API to infer structured fields from the dropped headlines.
- Clean and validate the LLM-generated outputs.
- Reintegrate the recovered rows into the final dataset.

---

## ✅ Outcome of Phase 1

We now have:
- A clean breakdown of what was dropped.
- A standalone file (`heards_dropped_rows.csv`) for LLM processing.

Proceed to Phase 2 when ready.


In [3]:
from google.colab import files
import pandas as pd

# Upload original dataset
print("📂 Please upload 'heards_full_dataset.csv'")
full_uploaded = files.upload()

# Upload cleaned dataset
print("📂 Now upload 'heards_structured_cleaned_03.csv'")
cleaned_uploaded = files.upload()


📂 Please upload 'heards_full_dataset.csv'


Saving heards_full_dataset.csv to heards_full_dataset.csv
📂 Now upload 'heards_structured_cleaned_03.csv'


Saving heards_structured_cleaned_03.csv to heards_structured_cleaned_03.csv


In [4]:
# Load both datasets
full_df = pd.read_csv("heards_full_dataset.csv")
cleaned_df = pd.read_csv("heards_structured_cleaned_03.csv")

# Show shape and columns for verification
print("🗃️ Full dataset:", full_df.shape)
print("🧼 Cleaned dataset:", cleaned_df.shape)
print("\n📌 Columns in cleaned dataset:")
print(cleaned_df.columns.tolist())


🗃️ Full dataset: (215400, 3)
🧼 Cleaned dataset: (178357, 16)

📌 Columns in cleaned dataset:
['updatedDate', 'start_date', 'end_date', 'action_type', 'party', 'price', 'price_basis', 'grade', 'incoterm', 'location', 'volume_min', 'volume_max', 'frequency_tag', 'source', 'headline', 'id']


In [5]:
# Filter out rows that were dropped (based on 'headline' mismatch)
dropped_df = full_df[~full_df['headline'].isin(cleaned_df['headline'])].reset_index(drop=True)

# Keep updatedDate as requested
print("🗃️ Dropped rows recovered:", dropped_df.shape)
print("📌 Columns:", dropped_df.columns.tolist())


🗃️ Dropped rows recovered: (37043, 3)
📌 Columns: ['headline', 'id', 'updatedDate']


In [6]:
# Drop 'id' column from cleaned and dropped datasets (if it exists)
if "id" in cleaned_df.columns:
    cleaned_df = cleaned_df.drop(columns=["id"])

if "id" in dropped_df.columns:
    dropped_df = dropped_df.drop(columns=["id"])

# Confirm column alignment
print("🧼 Cleaned columns:", cleaned_df.columns.tolist())
print("🗃️ Dropped columns:", dropped_df.columns.tolist())


🧼 Cleaned columns: ['updatedDate', 'start_date', 'end_date', 'action_type', 'party', 'price', 'price_basis', 'grade', 'incoterm', 'location', 'volume_min', 'volume_max', 'frequency_tag', 'source', 'headline']
🗃️ Dropped columns: ['headline', 'updatedDate']


In [7]:
print("🔎 Sample row from cleaned_df:")
display(cleaned_df.head(1))

print("\n🔎 Sample row from dropped_df:")
display(dropped_df.head(1))


🔎 Sample row from cleaned_df:


,updatedDate,start_date,end_date,action_type,party,price,price_basis,grade,incoterm,location,volume_min,volume_max,frequency_tag,source,headline
0,2025-05-23T14:44:56.674Z,2025-06-02,2025-06-06,Offer,GLTD,16.0,3,NaN,CIF,Malta,NaN,NaN,Any Day,regex,"Platts HSFO Med Crg CIF bss Malta 10-25, GLTD ..."



🔎 Sample row from dropped_df:


,headline,updatedDate
0,Platts Singapore Fuel Oil Bids Offers Trades,2025-05-23T11:03:11Z


In [8]:
# Split dropped_df into two halves
mid = len(dropped_df) // 2
dropped_half1 = dropped_df.iloc[:mid].reset_index(drop=True)
dropped_half2 = dropped_df.iloc[mid:].reset_index(drop=True)

# Save second half for later
dropped_half2.to_csv("heards_dropped_later.csv", index=False)

print("✅ Dropped rows split:")
print("➡️ dropped_half1:", dropped_half1.shape)
print("➡️ dropped_half2:", dropped_half2.shape, "(saved)")


✅ Dropped rows split:
➡️ dropped_half1: (18521, 2)
➡️ dropped_half2: (18522, 2) (saved)


In [9]:
!pip install python-dotenv


In [10]:
from google.colab import files
from dotenv import load_dotenv
import os

# Upload .env file again
print("📂 Please upload your .env file with API keys")
files.upload()

# Load API keys
load_dotenv(".env")
chatgpt_api_key = os.getenv("AI_API_KEY")
deepseek_api_key = os.getenv("SEEK_API_KEY")

if deepseek_api_key and chatgpt_api_key:
    print("✅ Both DeepSeek and ChatGPT API keys loaded successfully.")
else:
    raise ValueError("❌ One or both API keys are missing from .env.")


📂 Please upload your .env file with API keys


Saving .env to .env
✅ Both DeepSeek and ChatGPT API keys loaded successfully.


In [11]:
import requests
import json

def call_deepseek_api(headline: str, api_key: str) -> dict:
    prompt = f"""You are extracting structured trading data from a fuel oil market headline.

Extract the following fields:
- action_type (e.g., Bid, Offer, Raise, Lower, No Trade, No Bid, No Offer)
- price (float, if present)
- start_date (YYYY-MM-DD, if present)
- end_date (YYYY-MM-DD, if present)
- party (e.g., trader name)
- volume_min (int, if mentioned)
- volume_max (int, if mentioned)

Respond in JSON only. Do NOT use markdown formatting or explanations.

Headline: "{headline}"
"""

    url = "https://api.deepseek.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2
    }

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        message = response.json()["choices"][0]["message"]["content"]

        if message.strip().startswith("```json"):
            message = message.strip()[7:-3].strip()

        return json.loads(message)
    except Exception as e:
        print(f"❌ Error for headline: {headline[:60]}... — {e}")
        return {}


In [12]:
import time
import pandas as pd

# Final column structure (matching cleaned_df)
final_columns = [
    "updatedDate",
    "start_date",
    "end_date",
    "action_type",
    "party",
    "price",
    "price_basis",
    "grade",
    "incoterm",
    "location",
    "volume_min",
    "volume_max",
    "frequency_tag",
    "source",
    "headline"
]

# Function to create an empty row template
def empty_row_template():
    return {col: None for col in final_columns}

# Storage
results = []
checkpoint_path = "deepseek_test_output.csv"

# Sample 10 random rows
sample_rows = dropped_half1.sample(n=10, random_state=42).reset_index(drop=True)

# Inference loop
for i, row in sample_rows.iterrows():
    headline = row["headline"]
    updatedDate = row["updatedDate"]

    parsed = call_deepseek_api(headline, deepseek_api_key)

    filled = empty_row_template()
    filled.update(parsed)
    filled["headline"] = headline
    filled["updatedDate"] = updatedDate
    filled["source"] = "deepseek"

    results.append(filled)

    print(f"✅ Processed row {i+1}/10")
    time.sleep(1)  # throttle to avoid rate limits

# Save result
test_df = pd.DataFrame(results)[final_columns]
test_df.to_csv(checkpoint_path, index=False)
print(f"🧪 Test complete. Saved to '{checkpoint_path}'")


✅ Processed row 1/10
✅ Processed row 2/10
✅ Processed row 3/10
✅ Processed row 4/10
✅ Processed row 5/10
✅ Processed row 6/10
✅ Processed row 7/10
✅ Processed row 8/10
✅ Processed row 9/10
✅ Processed row 10/10
🧪 Test complete. Saved to 'deepseek_test_output.csv'


In [13]:
from google.colab import files
files.download("deepseek_test_output.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 🧹 HEARDS Dataset Filtering – Non-Informative Headlines

## 🧠 Context

During DeepSeek-based information extraction on the 37,043 dropped rows, we observed that a large portion of rows returned `null` or unstructured outputs. After reviewing randomly sampled headlines, we found that many of these rows were **non-informative notices** rather than actual bid/offer/trade data.

## 🕵️‍♂️ Problem

Common patterns among these headlines include:

- "Platts would like to clarify..."
- "Platts reserves the right to..."
- "Platts is aware that..."
- "Platts reminds participants..."

These are not market actions but administrative or legal disclaimers.

## 📉 Impact

These rows are not suitable for structured extraction (e.g., action_type, price, volume) and can safely be excluded from model inference. Initial analysis showed that a significant portion of the 17.2% dropped rows fall under these patterns.

## ✅ Solution

We filtered `dropped_half1` using a regular expression to remove rows matching the above patterns. The same filter will also be applied to `dropped_half2`.

This pre-filtering step ensures:
- Higher model precision
- Lower API costs
- Cleaner post-processing

## 🔍 Regex Pattern Used

```python
r"Platts (would like|reserves|is aware|reminds)"


In [14]:
import pandas as pd

# Define non-informative pattern (non-capturing group to suppress warnings)
non_info_pattern = r"Platts (?:would like|reserves|is aware|reminds)"

# --- Extract non-informative rows ---
non_info_half1 = dropped_half1[dropped_half1["headline"].str.contains(non_info_pattern, case=False, na=False, regex=True)]
non_info_half2 = dropped_half2[dropped_half2["headline"].str.contains(non_info_pattern, case=False, na=False, regex=True)]

# --- Filter them out from original datasets ---
filtered_half1 = dropped_half1[~dropped_half1["headline"].str.contains(non_info_pattern, case=False, na=False, regex=True)]
filtered_half2 = dropped_half2[~dropped_half2["headline"].str.contains(non_info_pattern, case=False, na=False, regex=True)]


non_info_half1.to_csv("heards_noninformative_half1.csv", index=False)
non_info_half2.to_csv("heards_noninformative_half2.csv", index=False)
filtered_half1.to_csv("heards_informative_half1.csv", index=False)
filtered_half2.to_csv("heards_informative_half2.csv", index=False)

# --- Summary ---
print("✅ Filtering complete.")
print("🧹 Non-informative half 1:", non_info_half1.shape)
print("🧹 Non-informative half 2:", non_info_half2.shape)
print("✅ Informative half 1:", filtered_half1.shape)
print("✅ Informative half 2:", filtered_half2.shape)


✅ Filtering complete.
🧹 Non-informative half 1: (2549, 2)
🧹 Non-informative half 2: (1708, 2)
✅ Informative half 1: (15972, 2)
✅ Informative half 2: (16814, 2)


In [15]:
import time
import pandas as pd

# Final output schema (matching cleaned_df minus 'id')
final_columns = [
    "updatedDate",
    "start_date",
    "end_date",
    "action_type",
    "party",
    "price",
    "price_basis",
    "grade",
    "incoterm",
    "location",
    "volume_min",
    "volume_max",
    "frequency_tag",
    "source",
    "headline"
]

# Empty row template
def empty_row_template():
    return {col: None for col in final_columns}

# Sample 10 random informative rows
sample_rows = filtered_half1.sample(n=10, random_state=42).reset_index(drop=True)
results = []

# Inference loop
for i, row in sample_rows.iterrows():
    headline = row["headline"]
    updatedDate = row["updatedDate"]

    parsed = call_deepseek_api(headline, deepseek_api_key)

    filled = empty_row_template()
    filled.update(parsed)
    filled["headline"] = headline
    filled["updatedDate"] = updatedDate
    filled["source"] = "deepseek"

    results.append(filled)

    print(f"✅ Processed row {i+1}/10")
    time.sleep(1)  # rate limiting

# Save test output
test_df = pd.DataFrame(results)[final_columns]
test_df.to_csv("deepseek_filtered_half1_TEST.csv", index=False)
print("🧪 Test complete. File saved as 'deepseek_filtered_half1_TEST.csv'")


✅ Processed row 1/10
✅ Processed row 2/10
✅ Processed row 3/10
✅ Processed row 4/10
✅ Processed row 5/10
✅ Processed row 6/10
✅ Processed row 7/10
✅ Processed row 8/10
✅ Processed row 9/10
✅ Processed row 10/10
🧪 Test complete. File saved as 'deepseek_filtered_half1_TEST.csv'


In [16]:
from google.colab import files
files.download("deepseek_filtered_half1_TEST.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# Step 1: Combine both filtered halves
combined_filtered = pd.concat([filtered_half1, filtered_half2], ignore_index=True)

# Step 2: Apply secondary regex-based filtering (broader junk patterns)
secondary_pattern = r"(Platts )?(reminds|is aware|would like|reserves|submit nominations|parties to|participants are|Platts encourages|may not be consistent|please contact|call for feedback|notice of|reminder:|correction:|statement:)"

refined_filtered = combined_filtered[~combined_filtered["headline"].str.contains(
    secondary_pattern, case=False, na=False, regex=True
)]

# Step 3: Show counts
print("🧮 After combining both halves:", combined_filtered.shape)
print("✅ After second-layer regex filtering:", refined_filtered.shape)


<ipython-input-17-1afd473a5ed0>:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  refined_filtered = combined_filtered[~combined_filtered["headline"].str.contains(


🧮 After combining both halves: (32786, 2)
✅ After second-layer regex filtering: (30748, 2)


In [18]:
# Randomly sample 20 rows from the cleaned dataset
sample_to_inspect = refined_filtered.sample(n=20, random_state=99).reset_index(drop=True)

# Show only the headline column for inspection
sample_to_inspect[["headline"]]


,headline
0,Asia 1572: PLATTS SINGAPORE FUEL OIL PAPER TRA...
1,Asia 1611: PLATTS HSFO: PHYSICAL BIDS FINALS O...
2,PLATTS HSFO: PHYSICAL OFFERS FINALS ON CLOSE (...
3,Platts Singapore Fuel Oil Bids Offers Trades
4,Asia 190: Platts HSFO: Platts accepts informat...
5,PLATTS HSFO FOB FUJ: PHYSICAL BIDS FINALS ON C...
6,Asia 1479: PLATTS HSFO: PHYSICAL BIDS FINALS O...
7,Asia 1820: PLATTS HSFO: PHYSICAL OFFERS FINALS...
8,PLATTS HSFO: PLATTS HSFO: PHYSICAL BIDS FINALS...
9,Platts HSFO/MF0.5% physical: The Straits Fuel ...


In [19]:
# Apply third-layer refinement to remove additional noise
third_pattern = r"(no trade|no bids?|no offers?|eWindow|facilitated through|submit.*instant messenger|information provided for publication|summary -)"
refined_filtered_final = refined_filtered[~refined_filtered["headline"].str.contains(
    third_pattern, case=False, na=False, regex=True
)]

# Show the change
print("🧮 After 2nd-layer filtering:", refined_filtered.shape)
print("✅ After 3rd-layer filtering:", refined_filtered_final.shape)


<ipython-input-19-a69e50087d2c>:3: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  refined_filtered_final = refined_filtered[~refined_filtered["headline"].str.contains(


🧮 After 2nd-layer filtering: (30748, 2)
✅ After 3rd-layer filtering: (17315, 2)


In [20]:
import time
import pandas as pd

# Final schema (aligned with cleaned_df, minus 'id')
final_columns = [
    "updatedDate",
    "start_date",
    "end_date",
    "action_type",
    "party",
    "price",
    "price_basis",
    "grade",
    "incoterm",
    "location",
    "volume_min",
    "volume_max",
    "frequency_tag",
    "source",
    "headline"
]

# Empty row template
def empty_row_template():
    return {col: None for col in final_columns}

# Sample 10 random rows from final refined set
sample_rows = refined_filtered_final.sample(n=10, random_state=42).reset_index(drop=True)
results = []

# Inference loop
for i, row in sample_rows.iterrows():
    headline = row["headline"]
    updatedDate = row["updatedDate"]

    parsed = call_deepseek_api(headline, deepseek_api_key)

    filled = empty_row_template()
    filled.update(parsed)
    filled["headline"] = headline
    filled["updatedDate"] = updatedDate
    filled["source"] = "deepseek"

    results.append(filled)
    print(f"✅ Processed row {i+1}/10")
    time.sleep(1)

# Save output
test_df = pd.DataFrame(results)[final_columns]
test_df.to_csv("deepseek_final_refined_TEST.csv", index=False)
print("🧪 Test complete. Saved to 'deepseek_final_refined_TEST.csv'")


✅ Processed row 1/10
✅ Processed row 2/10
✅ Processed row 3/10
✅ Processed row 4/10
✅ Processed row 5/10
✅ Processed row 6/10
✅ Processed row 7/10
✅ Processed row 8/10
✅ Processed row 9/10
✅ Processed row 10/10
🧪 Test complete. Saved to 'deepseek_final_refined_TEST.csv'


In [21]:
from google.colab import files
files.download("deepseek_final_refined_TEST.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
import time, json, os
import pandas as pd
from pathlib import Path

# ======= Config =======
batch_size = 5
row_limit = len(refined_filtered_final)
checkpoint_interval = 100
checkpoint_path = "deepseek_checkpoint.csv"
final_output_path = "deepseek_train_subset.csv"
key_columns = ["action_type", "price", "party"]
start_batch = 0  # Automatically updated later

# ======= Columns =======
final_columns = [
    "updatedDate", "start_date", "end_date", "action_type", "party", "price",
    "price_basis", "grade", "incoterm", "location", "volume_min",
    "volume_max", "frequency_tag", "source", "headline"
]

# ======= Helpers =======
def batch_prompt(headlines):
    lines = [f"{i+1}. \"{h}\"" for i, h in enumerate(headlines)]
    return f"""Extract the following fields from each oil market headline:

- action_type
- price
- start_date
- end_date
- party
- volume_min
- volume_max

Respond in JSON array format. One object per headline. Use null if not applicable. No explanations.

Headlines:
{chr(10).join(lines)}
"""

def parse_response(text, headlines, updatedDates):
    try:
        if text.strip().startswith("```json"):
            text = text.strip()[7:-3].strip()
        data = json.loads(text)
        rows = []
        for i, item in enumerate(data):
            row = {"headline": headlines[i], "updatedDate": updatedDates[i], "source": "deepseek"}
            row.update(item)
            for col in final_columns:
                row.setdefault(col, None)
            rows.append(row)
        return rows
    except Exception as e:
        print(f"❌ Parse error: {e}")
        return []

# ======= Load Prior Progress (if exists) =======
if os.path.exists(checkpoint_path):
    saved_df = pd.read_csv(checkpoint_path)
    results = saved_df.to_dict("records")
    start_batch = len(saved_df) // batch_size
    print(f"🔁 Resuming from batch {start_batch} — {len(saved_df)} rows loaded.")
else:
    results = []
    print("🚀 Starting fresh...")

# ======= Inference Loop =======
batches = (row_limit + batch_size - 1) // batch_size

for b in range(start_batch, batches):
    batch_df = refined_filtered_final.iloc[b*batch_size:(b+1)*batch_size]
    headlines = batch_df["headline"].tolist()
    dates = batch_df["updatedDate"].tolist()

    prompt = batch_prompt(headlines)
    payload = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.2
    }
    headers = {
        "Authorization": f"Bearer {deepseek_api_key}",
        "Content-Type": "application/json"
    }

    try:
        import requests
        response = requests.post("https://api.deepseek.com/v1/chat/completions", headers=headers, json=payload)
        response.raise_for_status()
        text = response.json()["choices"][0]["message"]["content"]
        parsed_rows = parse_response(text, headlines, dates)
        results.extend(parsed_rows)

        print(f"✅ Batch {b+1}/{batches} done — Total rows: {len(results)}")

        # Periodic Save
        if len(results) % checkpoint_interval == 0 or b == batches - 1:
            df = pd.DataFrame(results)[final_columns]
            df.dropna(subset=key_columns, how="all").to_csv(checkpoint_path, index=False)
            print(f"💾 Saved checkpoint at {len(results)} rows.")

    except Exception as e:
        print(f"❌ Failed at batch {b+1}: {e}")
        break

# ======= Final Save =======
df = pd.DataFrame(results)[final_columns]
filtered_df = df.dropna(subset=key_columns, how="all")
filtered_df.to_csv(final_output_path, index=False)
print(f"✅ Finished — {len(filtered_df)} rows saved to '{final_output_path}'")


🚀 Starting fresh...
✅ Batch 1/3463 done — Total rows: 5
✅ Batch 2/3463 done — Total rows: 10
✅ Batch 3/3463 done — Total rows: 15
✅ Batch 4/3463 done — Total rows: 20
✅ Batch 5/3463 done — Total rows: 25
✅ Batch 6/3463 done — Total rows: 30
✅ Batch 7/3463 done — Total rows: 35
✅ Batch 8/3463 done — Total rows: 40
✅ Batch 9/3463 done — Total rows: 45
✅ Batch 10/3463 done — Total rows: 50
✅ Batch 11/3463 done — Total rows: 55
✅ Batch 12/3463 done — Total rows: 60
✅ Batch 13/3463 done — Total rows: 65
✅ Batch 14/3463 done — Total rows: 70
✅ Batch 15/3463 done — Total rows: 75
✅ Batch 16/3463 done — Total rows: 80
✅ Batch 17/3463 done — Total rows: 85
✅ Batch 18/3463 done — Total rows: 90
✅ Batch 19/3463 done — Total rows: 95
✅ Batch 20/3463 done — Total rows: 100
💾 Saved checkpoint at 100 rows.
✅ Batch 21/3463 done — Total rows: 105
✅ Batch 22/3463 done — Total rows: 110
✅ Batch 23/3463 done — Total rows: 115
✅ Batch 24/3463 done — Total rows: 120
✅ Batch 25/3463 done — Total rows: 125
✅ 

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-23-1bcc33d79480>", line 88, in <cell line: 0>
    response = requests.post("https://api.deepseek.com/v1/chat/completions", headers=headers, json=payload)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/requests/api.py", line 115, in post
    return request("post", url, data=data, json=json, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/requests/api.py", line 59, in request
    return session.request(method=method, url=url, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/requests/sessions.py", line 589, in request
    resp

TypeError: object of type 'NoneType' has no len()

In [28]:
checkpoint_path = "deepseek_checkpoint.csv"
df = pd.read_csv(checkpoint_path)


In [29]:
final_output_path = "deepseek_train_subset.csv"
filtered_df.to_csv(final_output_path, index=False)


In [30]:
from google.colab import files
files.download("deepseek_train_subset.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
# Define keywords that typically indicate non-action headlines
non_action_keywords = [
    "expects", "forecast", "projection", "outlook", "trend", "analysis",
    "report", "market commentary", "anticipates", "likely", "seen", "view", "monitoring"
]

# Convert to lowercase for comparison
df_filtered = df[~df['headline'].str.lower().str.contains('|'.join(non_action_keywords), na=False)]

# View resulting shape
print(f"Original rows: {len(df)}")
print(f"Filtered rows: {len(df_filtered)}")

# Optional: Save filtered dataset
df_filtered.to_csv("deepseek_train_filtered.csv", index=False)


Original rows: 1501
Filtered rows: 1498


In [32]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

# Load your dataset
df = pd.read_csv("deepseek_train_subset.csv")

# Drop rows where all three key fields are missing
df = df.dropna(subset=['action_type', 'party', 'start_date'], how='all')

# ========== Action Type Model ==========
action_df = df.dropna(subset=['action_type'])
X_action = action_df['headline']
y_action = action_df['action_type']

X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_action, y_action, test_size=0.2, random_state=42)

action_model = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=500))
])


In [33]:
# Step 1: Import required libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
import joblib


In [34]:
# Step 2: Load cleaned dataset (03)
df_price = pd.read_csv("heards_structured_cleaned_03.csv")

# Drop any rows with missing price or headline
df_price = df_price.dropna(subset=["headline", "price"])

print(f"✅ Loaded {len(df_price)} rows with non-null 'headline' and 'price'")


✅ Loaded 178357 rows with non-null 'headline' and 'price'


In [35]:
# Step 3: Define text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)  # Remove URLs
    text = re.sub(r"[^a-z\s]", "", text)       # Remove non-letter characters
    text = re.sub(r"\s+", " ", text).strip()   # Normalize whitespace
    return text


In [36]:
# Step 4: Apply text cleaning and split dataset
df_price["clean_headline"] = df_price["headline"].apply(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df_price["clean_headline"],
    df_price["price"],
    test_size=0.1,
    random_state=42
)


In [37]:
# Step 5: TF-IDF vectorization and model training
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = Ridge(alpha=1.0)
model.fit(X_train_vec, y_train)

# Save model and vectorizer
joblib.dump(model, "price_predictor_ridge.joblib")
joblib.dump(vectorizer, "tfidf_vectorizer.joblib")
print("✅ Model and vectorizer saved.")


✅ Model and vectorizer saved.


In [38]:
# Step 6: Evaluate model performance
preds = model.predict(X_test_vec)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"📊 MAE (Mean Absolute Error): {mae:.2f}")
print(f"📈 R² Score: {r2:.2f}")


📊 MAE (Mean Absolute Error): 15.17
📈 R² Score: 0.95


In [39]:
# Step 1: Load CSV and parse dates
df_date = pd.read_csv("heards_structured_cleaned_03.csv", parse_dates=["updatedDate", "start_date"])

# Drop rows with missing dates
df_date = df_date.dropna(subset=["updatedDate", "start_date", "headline"])
print(f"✅ {len(df_date)} rows with non-null updatedDate and start_date.")


✅ 178357 rows with non-null updatedDate and start_date.


In [41]:
# Ensure both date columns are parsed as datetime
df_date["updatedDate"] = pd.to_datetime(df_date["updatedDate"], errors="coerce")
df_date["start_date"] = pd.to_datetime(df_date["start_date"], errors="coerce")

# Now drop rows with invalid dates
df_date = df_date.dropna(subset=["updatedDate", "start_date"])


In [46]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

# === Step 1: Load and Clean Training Data ===
df_date = pd.read_csv("heards_structured_cleaned_03.csv", parse_dates=["updatedDate", "start_date"])
df_date = df_date.dropna(subset=["updatedDate", "start_date", "headline"])

# Convert to tz-naive using mixed format
df_date["updatedDate"] = pd.to_datetime(df_date["updatedDate"], format='mixed', errors="coerce").dt.tz_localize(None)
df_date["start_date"] = pd.to_datetime(df_date["start_date"], format='mixed', errors="coerce").dt.tz_localize(None)
df_date = df_date.dropna(subset=["updatedDate", "start_date"])

print(f"✅ {len(df_date)} valid training rows before filtering")

# === Step 2: Create Target Variable (days between dates) ===
df_date.loc[:, "days_diff"] = (df_date["start_date"] - df_date["updatedDate"]).dt.days
df_date = df_date[df_date["days_diff"].between(-90, 90)]
print(f"🧹 Filtered to {len(df_date)} rows with realistic date gaps (±90 days)")

# === Step 3: Vectorize Headline Text ===
vectorizer_date = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X = vectorizer_date.fit_transform(df_date["headline"])
y = df_date["days_diff"]

# === Step 4: Train-Test Split and Model Training ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_date = Ridge(alpha=1.0)
model_date.fit(X_train, y_train)

# === Step 5: Evaluate Model ===
y_pred = model_date.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"📊 MAE on test set: {mae:.2f} days")

# === Step 6: Inference Function for Integration ===
def predict_start_date(headline, updated_date_str):
    try:
        updated_date = pd.to_datetime(updated_date_str, format='mixed', errors='coerce')
        if pd.isna(updated_date):
            return None

        vec = vectorizer_date.transform([headline])
        pred_days = model_date.predict(vec)[0]
        predicted_date = updated_date + pd.to_timedelta(round(pred_days), unit="D")

        # Drop if misleading
        if pd.Timestamp("2010-01-01") < predicted_date < pd.Timestamp("2030-01-01") and abs(pred_days) <= 90:
            return predicted_date
    except:
        pass
    return None


✅ 178251 valid training rows before filtering
🧹 Filtered to 8073 rows with realistic date gaps (±90 days)


<ipython-input-46-97f4019b513a>:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_date.loc[:, "days_diff"] = (df_date["start_date"] - df_date["updatedDate"]).dt.days


📊 MAE on test set: 2.87 days


In [51]:
import pandas as pd

# === Load cleaned dropped dataset ===
try:
    df_dropped = refined_filtered_final.copy()
except NameError:
    df_dropped = pd.read_csv("heards_informative_final.csv")

# === Preprocess ===
df_dropped["headline"] = df_dropped["headline"].astype(str).str.lower()
df_dropped["updatedDate"] = pd.to_datetime(df_dropped["updatedDate"], format='mixed', errors="coerce")
df_dropped = df_dropped.dropna(subset=["headline", "updatedDate"]).reset_index(drop=True)
print(f"🧾 Using {len(df_dropped)} dropped rows for inference")

# === ACTION TYPE MODEL ONLY ===
def predict_action_type(text):
    try:
        vec = vectorizer_action.transform([text])
        return model_action.predict(vec)[0]
    except:
        return None

# === Apply prediction ===
print("🔍 Predicting action_type...")
df_dropped["pred_action_type"] = df_dropped["headline"].apply(predict_action_type)

# === Diagnostic info ===
num_predicted = df_dropped["pred_action_type"].notna().sum()
print(f"✅ Action type predicted for {num_predicted} rows out of {len(df_dropped)}")

# === Save intermediate result for next step ===
df_dropped.to_csv("step1_action_type_output.csv", index=False)
print("💾 Saved to: step1_action_type_output.csv")

# === Preview sample ===
df_dropped[["headline", "pred_action_type"]].head(10)


🧾 Using 17315 dropped rows for inference
🔍 Predicting action_type...
✅ Action type predicted for 0 rows out of 17315
💾 Saved to: step1_action_type_output.csv


,headline,pred_action_type
0,platts singapore fuel oil bids offers trades,None
1,platts hsfo: physical offers finals on close (...,None
2,platts hsfo: platts hsfo: physical bids finals...,None
3,platts hsfo: physical offers finals on close (...,None
4,platts singapore fuel oil bids offers trades,None
5,platts bunker: 380 cst hsfo: delivered colombo...,None
6,platts hsfo: physical offers finals on close (...,None
7,platts hsfo: platts hsfo: physical bids finals...,None
8,platts hsfo: physical offers finals on close (...,None
9,platts singapore fuel oil bids offers trades,None


In [52]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib

# === STEP 1: Load structured data ===
df = pd.read_csv("heards_structured_cleaned_03.csv", parse_dates=["updatedDate", "start_date"])

# === STEP 2: Drop rows missing target or headline ===
df = df.dropna(subset=["action_type", "headline"])

# === STEP 3: Normalize text ===
df["headline"] = df["headline"].astype(str).str.lower()

# === STEP 4: Vectorize ===
vectorizer_action = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer_action.fit_transform(df["headline"])
y = df["action_type"]

# === STEP 5: Train/Test Split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === STEP 6: Train model ===
model_action = LogisticRegression(max_iter=200, random_state=42)
model_action.fit(X_train, y_train)

# === STEP 7: Evaluate ===
y_pred = model_action.predict(X_test)
print("📊 Action Type Classification Report:")
print(classification_report(y_test, y_pred))

# === STEP 8: Save artifacts ===
joblib.dump(model_action, "model_action.pkl")
joblib.dump(vectorizer_action, "vectorizer_action.pkl")
print("✅ Model and vectorizer saved.")


📊 Action Type Classification Report:
               precision    recall  f1-score   support

          Bid       1.00      1.00      1.00      4872
        Lower       1.00      1.00      1.00     12063
     No Trade       1.00      1.00      1.00      3996
        Offer       1.00      1.00      1.00      5142
        Raise       1.00      1.00      1.00      9590
Trade Summary       1.00      0.56      0.71         9

     accuracy                           1.00     35672
    macro avg       1.00      0.93      0.95     35672
 weighted avg       1.00      1.00      1.00     35672

✅ Model and vectorizer saved.


In [54]:
import pandas as pd
import joblib

# === Load retrained model and vectorizer ===
model_action = joblib.load("model_action.pkl")
vectorizer_action = joblib.load("vectorizer_action.pkl")

# === Use in-memory dropped dataset ===
df_dropped = refined_filtered_final.copy()

# === Clean ===
df_dropped["headline"] = df_dropped["headline"].astype(str).str.lower()
df_dropped["updatedDate"] = pd.to_datetime(df_dropped["updatedDate"], format='mixed', errors="coerce")
df_dropped = df_dropped.dropna(subset=["headline", "updatedDate"]).reset_index(drop=True)
print(f"🧾 Using {len(df_dropped)} dropped rows for inference")

# === Action type prediction ===
def predict_action_type(text):
    try:
        vec = vectorizer_action.transform([text])
        return model_action.predict(vec)[0]
    except:
        return None

print("🔍 Predicting action_type...")
df_dropped["pred_action_type"] = df_dropped["headline"].apply(predict_action_type)

# === Summary ===
num_predicted = df_dropped["pred_action_type"].notna().sum()
print(f"✅ Action type predicted for {num_predicted} rows out of {len(df_dropped)}")

# === Save ===
df_dropped.to_csv("step1_action_type_output.csv", index=False)
print("💾 Saved to: step1_action_type_output.csv")


🧾 Using 17315 dropped rows for inference
🔍 Predicting action_type...
✅ Action type predicted for 17315 rows out of 17315
💾 Saved to: step1_action_type_output.csv


In [55]:
df_dropped[["headline", "pred_action_type"]].sample(10, random_state=42)


,headline,pred_action_type
169,"platts hsfo 380cst fob straits 15-30, trafi se...",Offer
8875,platts hsfo: physical offers finals on close (...,Offer
16789,asia 1757: platts hsfo 380cst fob straits 15-3...,Bid
2267,platts hsfo: platts hsfo: physical bids finals...,Bid
15927,asia 1911: platts hsfo: physical bids finals o...,Bid
9304,platts singapore fuel oil bids offers trades,Offer
828,platts hsfo: physical bids finals on close (18...,Bid
7975,platts hsfo: physical offers finals on close (...,Offer
15088,asia 427: platts hsfo: physical offers summary...,Offer
14789,asia 1955: platts asia deals summary: refined ...,Offer


In [59]:
from sklearn.linear_model import Ridge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd
import joblib

# === Load dataset ===
df_price = pd.read_csv("heards_structured_cleaned_03.csv")
df_price = df_price.dropna(subset=["price", "headline"])
df_price = df_price[df_price["price"].between(10, 1000)]  # realistic filter

# === Preprocess ===
df_price["headline"] = df_price["headline"].astype(str).str.lower()

# === Features/labels ===
X = df_price["headline"]
y = df_price["price"]

# === Vectorizer (likely previous settings) ===
vectorizer_price = TfidfVectorizer(max_features=8000, ngram_range=(1, 3))
X_vec = vectorizer_price.fit_transform(X)

# === Split ===
X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

# === Train model ===
model_price = Ridge(alpha=0.5)  # try previous alpha
model_price.fit(X_train, y_train)

# === Evaluate ===
preds = model_price.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)
print(f"📊 MAE (Mean Absolute Error): {mae:.2f}")
print(f"📈 R² Score: {r2:.2f}")

# === Save if MAE is close to 15 ===
joblib.dump(model_price, "model_price.pkl")
joblib.dump(vectorizer_price, "vectorizer_price.pkl")
print("✅ High-accuracy price model saved.")


📊 MAE (Mean Absolute Error): 16.36
📈 R² Score: 0.97
✅ High-accuracy price model saved.


In [60]:
import pandas as pd
import joblib
import numpy as np

# === Load saved model & vectorizer ===
model_price = joblib.load("model_price.pkl")
vectorizer_price = joblib.load("vectorizer_price.pkl")

# === Load previous output from Step 1 ===
df = pd.read_csv("step1_action_type_output.csv")

# === Clean headlines ===
df["headline"] = df["headline"].astype(str).str.lower()

# === Predict price ===
def predict_price(text):
    try:
        vec = vectorizer_price.transform([text])
        return float(model_price.predict(vec)[0])
    except:
        return np.nan

print("🔍 Predicting price...")
df["pred_price"] = df["headline"].apply(predict_price)

# === Summary ===
predicted = df["pred_price"].notna().sum()
print(f"✅ Price predicted for {predicted} rows out of {len(df)}")

# === Save result ===
df.to_csv("step2_price_output.csv", index=False)
print("💾 Saved to: step2_price_output.csv")


🔍 Predicting price...
✅ Price predicted for 17315 rows out of 17315
💾 Saved to: step2_price_output.csv


In [63]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

# === Step 1: Load cleaned dataset ===
df_date = pd.read_csv("heards_structured_cleaned_03.csv")

# === Step 2: Parse and sanitize dates ===
df_date["updatedDate"] = pd.to_datetime(df_date["updatedDate"], format="mixed", errors="coerce", utc=True).dt.tz_localize(None)
df_date["start_date"] = pd.to_datetime(df_date["start_date"], format="mixed", errors="coerce", utc=True).dt.tz_localize(None)

# === Step 3: Drop rows with missing critical fields ===
df_date = df_date.dropna(subset=["headline", "updatedDate", "start_date"])
print(f"✅ {len(df_date)} valid training rows before filtering")

# === Step 4: Compute difference in days ===
df_date["days_diff"] = (df_date["start_date"] - df_date["updatedDate"]).dt.days

# === Step 5: Filter to realistic date differences (e.g., ±90 days) ===
df_date = df_date[(df_date["days_diff"] >= -90) & (df_date["days_diff"] <= 90)]
print(f"🧹 Filtered to {len(df_date)} rows with realistic date gaps (±90 days)")

# === Step 6: Prepare training data ===
X = df_date["headline"].astype(str).str.lower()
y = df_date["days_diff"]

vectorizer = TfidfVectorizer(min_df=3, max_features=5000, ngram_range=(1,2))
X_vec = vectorizer.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_vec, y, test_size=0.2, random_state=42)

# === Step 7: Train model ===
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# === Step 8: Evaluate ===
preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"📊 MAE (Mean Absolute Error): {mae:.2f}")
print(f"📈 R² Score: {r2:.2f}")

# === Step 9: Save model and vectorizer ===
joblib.dump(model, "model_days_diff.pkl")
joblib.dump(vectorizer, "vectorizer_days_diff.pkl")
print("✅ Date model and vectorizer saved.")


✅ 178251 valid training rows before filtering
🧹 Filtered to 8073 rows with realistic date gaps (±90 days)


<ipython-input-63-41763aba1d8b>:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_date["days_diff"] = (df_date["start_date"] - df_date["updatedDate"]).dt.days


📊 MAE (Mean Absolute Error): 2.91
📈 R² Score: 0.45
✅ Date model and vectorizer saved.


In [64]:
import pandas as pd
import joblib
import numpy as np
from datetime import timedelta

# === Load model and vectorizer ===
model_date = joblib.load("model_days_diff.pkl")
vectorizer_date = joblib.load("vectorizer_days_diff.pkl")

# === Load previous output ===
df = pd.read_csv("step2_price_output.csv")

# === Clean headlines ===
df["headline"] = df["headline"].astype(str).str.lower()

# === Predict days_diff ===
def predict_days_diff(text):
    try:
        vec = vectorizer_date.transform([text])
        return int(model_date.predict(vec)[0])
    except:
        return np.nan

print("🔍 Predicting start_date...")
df["pred_days_diff"] = df["headline"].apply(predict_days_diff)

# === Parse updatedDate ===
df["updatedDate"] = pd.to_datetime(df["updatedDate"], errors="coerce", format='mixed')

# === Compute start_date ===
df["pred_start_date"] = df.apply(
    lambda row: row["updatedDate"] + timedelta(days=row["pred_days_diff"])
    if pd.notnull(row["updatedDate"]) and pd.notnull(row["pred_days_diff"])
    else None,
    axis=1
)

# === Summary ===
predicted = df["pred_start_date"].notna().sum()
print(f"✅ start_date predicted for {predicted} rows out of {len(df)}")

# === Save final output ===
df.to_csv("heards_model_predicted_final.csv", index=False)
print("💾 Final file saved to: heards_model_predicted_final.csv")


🔍 Predicting start_date...
✅ start_date predicted for 17315 rows out of 17315
💾 Final file saved to: heards_model_predicted_final.csv


In [69]:
import pandas as pd

# === Load Cleaned V3 Dataset ===
df_cleaned = pd.read_csv("heards_structured_cleaned_03.csv")

# === Load Recovered Predictions ===
df_predicted = pd.read_csv("heards_model_predicted_final.csv")

# === Standardize Column Names If Needed ===
# Make sure the predicted columns align with the cleaned dataset
df_predicted.rename(columns={
    "pred_action_type": "action_type",
    "pred_price": "price",
    "pred_start_date": "start_date"
}, inplace=True)

# === Concatenate Datasets ===
df_combined = pd.concat([df_cleaned, df_predicted], ignore_index=True)

# === Save as Version 04 ===
df_combined.to_csv("heards_structured_cleaned_04.csv", index=False)
print("✅ Merged dataset saved to: heards_structured_cleaned_04.csv")
print(f"🧾 Final row count: {len(df_combined)}")


✅ Merged dataset saved to: heards_structured_cleaned_04.csv
🧾 Final row count: 195672


In [70]:
import pandas as pd

# === Load Merged Dataset ===
df = pd.read_csv("heards_structured_cleaned_04.csv")

# === Sanity Check ===
summary = pd.DataFrame({
    "Non-Null Count": df.notnull().sum(),
    "Total Rows": len(df),
    "% Filled": (df.notnull().sum() / len(df) * 100).round(2),
    "Data Type": df.dtypes
})

# === Display Sorted by % Filled ===
summary = summary.sort_values(by="% Filled", ascending=False)
print("🧠 Sanity Check: Column Fill Rates & Types")
print(summary)


🧠 Sanity Check: Column Fill Rates & Types
                Non-Null Count  Total Rows  % Filled Data Type
updatedDate             195672      195672    100.00    object
start_date              195672      195672    100.00    object
action_type             195672      195672    100.00    object
price                   195672      195672    100.00   float64
headline                195672      195672    100.00    object
party                   178357      195672     91.15    object
id                      178357      195672     91.15    object
source                  178357      195672     91.15    object
end_date                178357      195672     91.15    object
grade                   174502      195672     89.18   float64
location                161898      195672     82.74    object
incoterm                160850      195672     82.20    object
price_basis             158335      195672     80.92    object
volume_min              119534      195672     61.09   float64
volume_max   

In [71]:
import pandas as pd

# === Load Rebuilt Dataset ===
df = pd.read_csv("heards_structured_cleaned_04.csv")

# === Convert Dates to datetime ===
date_cols = ["updatedDate", "start_date", "end_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="ISO8601", errors="coerce", utc=True).dt.tz_localize(None)

# === Verify Changes ===
print("🧾 Final Data Types:")
print(df[date_cols].dtypes)

print("\n🔍 Null Counts:")
for col in date_cols:
    print(f"{col}: {df[col].isnull().sum()}")

# === Save Final Output ===
df.to_csv("heards_structured_cleaned_04.csv", index=False)
print("✅ Cleaned dataset overwritten to: heards_structured_cleaned_04.csv")


🧾 Final Data Types:
updatedDate    datetime64[ns]
start_date     datetime64[ns]
end_date       datetime64[ns]
dtype: object

🔍 Null Counts:
updatedDate: 0
start_date: 106
end_date: 17409
✅ Cleaned dataset overwritten to: heards_structured_cleaned_04.csv


In [79]:
# === Re-run sanity check on cleaned dataset ===
total_rows = len(df)
sanity_check = pd.DataFrame({
    "Non-Null Count": df.notna().sum(),
    "Total Rows": total_rows,
    "% Filled": (df.notna().sum() / total_rows * 100).round(2),
    "Data Type": df.dtypes.astype(str)
})

sanity_check = sanity_check.reset_index().rename(columns={"index": "Column"})
print("🧠 Sanity Check: Column Fill Rates & Types")
display(sanity_check)


🧠 Sanity Check: Column Fill Rates & Types


,Column,Non-Null Count,Total Rows,% Filled,Data Type
0,updatedDate,195672,195672,100.00,object
1,start_date,195566,195672,99.95,object
2,end_date,178263,195672,91.10,object
3,action_type,195672,195672,100.00,object
4,party,178357,195672,91.15,object
5,price,195672,195672,100.00,float64
6,price_basis,158335,195672,80.92,object
7,grade,174502,195672,89.18,float64
8,incoterm,160850,195672,82.20,object
9,location,161898,195672,82.74,object
